In [ ]:
# notebook details
'''
This cell contains details about the folowing Kaggle notebook.

The base resource used in completion of this work is:
https://github.com/ManthanKPatel/Camouflage-Detection/blob/main/DLProject.ipynb

The aim of this notebook is to provide a benchmarking log for a baseline SINet model for an article "Modified Convolutional Architecture for Camouflaged Object Detection".

Baseline SINet paper:
D.-P. Fan, G.-P. Ji, G. Sun, M.-M. Cheng, J. Shen, and L. Shao, “Camouflaged object detection,” in Conference on computer vision and pattern recognition. Seattle, WA, USA: IEEE, 2020, pp. 2777–2787, DOI: 10.1109/CVPR42600.2020.00285.

Baseline SINet GitHub repository:
https://github.com/DengPingFan/SINet/

For Hybrid-SINet modification, refer to https://www.kaggle.com/code/ivanomelchenkoim11/article-hybrid-sinet-benchmark.
For BGNet benchmark, refer to https://www.kaggle.com/code/ivanomelchenkoim11/article-bgnet-benchmark.
For C2F-Net benchmark, refer to https://www.kaggle.com/code/ivanomelchenkoim11/article-c2f-net-benchmark.
For pretrained FSPNet benchmark, refer to https://www.kaggle.com/code/ivanomelchenkoim11/article-fspnet-benchmark-pretrained.

References:
    1. D.-P. Fan, G.-P. Ji, G. Sun, M.-M. Cheng, J. Shen, and L. Shao, “Camouflaged object detection,” in Conference on computer vision and pattern recognition. Seattle, WA, USA: IEEE, 2020, pp. 2777–2787, DOI: 10.1109/CVPR42600.2020.00285.
    2. Ji, G.-P., Fan, D.-P., Chou, Y.-C., Dai, D., Liniger, A., & Van Gool, L. (2022). Deep Gradient Learning for Efficient Camouflaged Object Detection. Computer Vision and Pattern Recognition. doi:10.48550/ARXIV.2205.12853  
    3. Liu, Y., Wang, C.-q., & Zhou, Y.-j. (2021). Camouflaged people detection based on a semi-supervised search identification network. Defence Technology. doi:https://doi.org/10.1016/j.dt.2021.09.004 
    4. Sun, Y., Wang, S., Chen, C., & Xiang, T. Z. (2022). Boundary-guided camouflaged object detection. arXiv preprint arXiv:2207.00794.
    5. H. Mei, G.-P. Ji, Z. Wei, X. Yang, X. Wei, and D.-P. Fan, “Camouflaged object segmentation with distraction mining,” in Conference on computer vision and pattern recognition. Nashville, TN, USA: IEEE, 2021, pp. 8772–8781, DOI: 10.1109/CVPR46437.2021.00866
    6. Lv, J. Zhang, Y. Dai, A. Li, B. Liu, N. Barnes, and D.-P. Fan, “Simultaneously localize, segment and rank the camouflaged objects,” Computer Vision and Pattern Recognition (CVPR), 2021.
    7. Yujia Sun, Geng Chen, Tao Zhou, Yi Zhang, and Nian Liu. Context-aware cross-level fusion network for camouflaged object detection. arXiv preprint arXiv:2105.12555, 2021
    8. Zhou Huang, Hang Dai, Tian-Zhu Xiang, Shuo Wang, Huai-Xin Chen, Jie Qin, and Huan Xiong. Feature shrinkage pyramid for camouflaged object detection with transformers. In Proceedings of the IEEE/CVF conference on computer vision and pattern recognition, pages 5557–5566, 2023.

The following datasets are added to this notebook for training and testing purposes:
- COD10K
- CAMO
- NC4K
- Military Personnel Data
The sources of the datasets are provided in their descriptions.
'''

This cell defines the ResNet-50 backbone modified into two parallel branches.
The first branch is used by the Search Module to find potential camouflaged regions,
and the second branch is used by the Identification Module to refine the object boundary.
This is basically backbone of SINet.


In [4]:
# cell:1 backbone

import torch.nn as nn
import math


def conv3x3(in_planes, out_planes, stride=1):
    """
    3x3 convolution with padding
    """
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class ResNet_2Branch(nn.Module):
    # ResNet50 with two branches (modified from torchvision.models.resnet (pytorch==0.4.1))
    def __init__(self):
        # self.inplanes = 128
        self.inplanes = 64
        super(ResNet_2Branch, self).__init__()

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(Bottleneck, 64, 3)
        self.layer2 = self._make_layer(Bottleneck, 128, 4, stride=2)
        self.layer3_1 = self._make_layer(Bottleneck, 256, 6, stride=2)
        self.layer4_1 = self._make_layer(Bottleneck, 512, 3, stride=2)

        self.inplanes = 512
        self.layer3_2 = self._make_layer(Bottleneck, 256, 6, stride=2)
        self.layer4_2 = self._make_layer(Bottleneck, 512, 3, stride=2)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x1 = self.layer3_1(x)
        x1 = self.layer4_1(x1)

        x2 = self.layer3_2(x)
        x2 = self.layer4_2(x2)

        return x1, x2

In [5]:
# cell:2 dataloader (corrected + augmentation toggle)

import os
from pathlib import Path
from PIL import Image
import torch
import torch.utils.data as data
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
import random

# ====== toggle ======
USE_AUG = True   # <- set False to disable augmentation for training

# ---------- helpers ----------
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

def list_files(root, exts=IMG_EXTS):
    return sorted([os.path.join(root, f) for f in os.listdir(root) if f.lower().endswith(exts)])

def pair_images_and_gts(img_paths, gt_paths):
    """Match by filename stem and keep only pairs with identical spatial size."""
    gt_map = {Path(p).stem: p for p in gt_paths}
    imgs, gts = [], []
    for ip in img_paths:
        stem = Path(ip).stem
        gp = gt_map.get(stem)
        if gp and os.path.exists(gp):
            with Image.open(ip) as im, Image.open(gp) as gm:
                if im.size == gm.size:
                    imgs.append(ip); gts.append(gp)
    return imgs, gts

# ---------- training dataset ----------
class CamObjDataset(data.Dataset):
    def __init__(self, image_root, gt_root, trainsize, use_aug=USE_AUG):
        self.trainsize = trainsize
        self.use_aug = use_aug

        img_paths = list_files(image_root, IMG_EXTS)
        gt_paths  = list_files(gt_root,  IMG_EXTS)
        self.images, self.gts = pair_images_and_gts(img_paths, gt_paths)
        self.size = len(self.images)

        # final transforms (after any augment)
        self.img_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize), interpolation=InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])
        self.gt_transform = transforms.Compose([
            transforms.Resize((self.trainsize, self.trainsize), interpolation=InterpolationMode.NEAREST),
            transforms.ToTensor()   # -> [0,1]
        ])

        # augmentation components (light & safe)
        self.cj = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02)

    def __getitem__(self, index):
        img = self.rgb_loader(self.images[index])
        gt  = self.binary_loader(self.gts[index])

        if self.use_aug:
            # --- paired horizontal flip ---
            if random.random() < 0.5:
                img = TF.hflip(img); gt = TF.hflip(gt)

            # --- paired light affine (same params for img & gt) ---
            # small rotations, translations, scaling, shear
            angle = random.uniform(-5, 5)                  # degrees
            translate = (random.uniform(-0.05, 0.05)*img.size[0],
                         random.uniform(-0.05, 0.05)*img.size[1])
            scale = random.uniform(0.9, 1.1)
            shear = random.uniform(-5, 5)

            img = TF.affine(img, angle=angle, translate=translate, scale=scale, shear=shear,
                            interpolation=InterpolationMode.BILINEAR)
            gt  = TF.affine(gt,  angle=angle, translate=translate, scale=scale, shear=shear,
                            interpolation=InterpolationMode.NEAREST)

            # --- color jitter on image only ---
            img = self.cj(img)

        # final resize + tensorize + norm
        img = self.img_transform(img)
        gt  = self.gt_transform(gt)
        gt  = (gt > 0.5).float()  # binarize to {0,1}

        return img, gt

    def __len__(self):
        return self.size

    @staticmethod
    def rgb_loader(path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')

    @staticmethod
    def binary_loader(path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('L')

# ---------- test dataset (batchsize=1 style) ----------
class TestDataset:
    """Load test dataset (iterative .load_data())."""
    def __init__(self, image_root, gt_root, testsize):
        self.testsize = testsize
        img_paths = list_files(image_root, IMG_EXTS)
        gt_paths  = list_files(gt_root,  IMG_EXTS)
        self.images, self.gts = pair_images_and_gts(img_paths, gt_paths)
        self.size = len(self.images)
        self.index = 0

        self.transform = transforms.Compose([
            transforms.Resize((self.testsize, self.testsize), interpolation=InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])
        self.gt_transform = transforms.Compose([
            transforms.Resize((self.testsize, self.testsize), interpolation=InterpolationMode.NEAREST),
            transforms.ToTensor()
        ])

    def load_data(self):
        image = self.rgb_loader(self.images[self.index])
        image = self.transform(image).unsqueeze(0)

        gt = self.binary_loader(self.gts[self.index])
        gt = self.gt_transform(gt)

        name = Path(self.images[self.index]).with_suffix('.png').name
        self.index += 1
        return image, gt, name

    @staticmethod
    def rgb_loader(path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')

    @staticmethod
    def binary_loader(path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('L')

# ---------- fast test (no GT) ----------
class test_loader_faster(data.Dataset):
    def __init__(self, image_root, testsize):
        self.testsize = testsize
        self.images = list_files(image_root, IMG_EXTS)
        self.transform = transforms.Compose([
            transforms.Resize((self.testsize, self.testsize), interpolation=InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])

    def __getitem__(self, index):
        img = self.rgb_loader(self.images[index])
        img = self.transform(img)
        return img, self.images[index]

    def __len__(self):
        return len(self.images)

    @staticmethod
    def rgb_loader(path):
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')

# ---------- loader factory ----------
def get_loader(image_root, gt_root, batchsize, trainsize, shuffle=True, num_workers=0, pin_memory=True):
    dataset = CamObjDataset(image_root, gt_root, trainsize, use_aug=USE_AUG)
    loader = data.DataLoader(dataset=dataset,
                             batch_size=batchsize,
                             shuffle=shuffle,
                             num_workers=num_workers,
                             pin_memory=pin_memory)
    return loader


In [6]:
# cell:3 trainer (modernized: BCE+Dice, val, save-best, AMP)

import os, math, torch, torch.nn.functional as F
from datetime import datetime
from dataclasses import dataclass

# ===== toggles =====
USE_DICE     = True     # add Dice loss with BCE for crisper masks
USE_AMP      = True     # mixed precision for speed/memory
SAVE_BEST    = True     # save best model by val MAE↓
BEST_FILENAME = "SINet_best.pth"

# ---------- losses ----------
class DiceLoss(torch.nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2.0 * (probs * targets).sum(dim=(2,3)) + self.eps
        den = (probs.pow(2) + targets.pow(2)).sum(dim=(2,3)) + self.eps
        return 1.0 - (num / den).mean()

def build_loss():
    bce = torch.nn.BCEWithLogitsLoss()
    if not USE_DICE:
        return bce
    dice = DiceLoss()
    def combo(logits, targets):
        return bce(logits, targets) + dice(logits, targets)
    return combo

# ---------- utils ----------
def clip_gradient(optimizer, grad_clip):
    if grad_clip is None or grad_clip <= 0: return
    for group in optimizer.param_groups:
        for p in group["params"]:
            if p.grad is not None:
                p.grad.data.clamp_(-grad_clip, grad_clip)

@torch.no_grad()
def eval_mae(pred, gt):
    return torch.abs(pred - gt).mean()

@dataclass
class TrainStats:
    loss_s: float = 0.0
    loss_i: float = 0.0
    steps:  int  = 0

# ---------- training / validation ----------
def train_one_epoch(train_loader, model, optimizer, epoch, opt, loss_func, scaler=None):
    model.train()
    stats = TrainStats()
    device = next(model.parameters()).device

    for step, (images, gts) in enumerate(train_loader, start=1):
        images = images.to(device, non_blocking=True)
        gts    = gts.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if USE_AMP and scaler is not None:
            with torch.cuda.amp.autocast():
                cam_sm, cam_im = model(images)
                loss_sm = loss_func(cam_sm, gts)
                loss_im = loss_func(cam_im, gts)
                loss_total = loss_sm + loss_im
            scaler.scale(loss_total).backward()
            clip_gradient(optimizer, opt.clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            cam_sm, cam_im = model(images)
            loss_sm = loss_func(cam_sm, gts)
            loss_im = loss_func(cam_im, gts)
            loss_total = loss_sm + loss_im
            loss_total.backward()
            clip_gradient(optimizer, opt.clip)
            optimizer.step()

        stats.loss_s += float(loss_sm.detach().cpu())
        stats.loss_i += float(loss_im.detach().cpu())
        stats.steps  += 1

        if step % 10 == 0 or step == len(train_loader):
            print(f"[{datetime.now()}] [Epoch {epoch:03d}/{opt.epoch:03d}] "
                  f"[Step {step:04d}/{len(train_loader):04d}] "
                  f"Loss_s={float(loss_sm):.4f} Loss_i={float(loss_im):.4f}")

    # epoch averages
    stats.loss_s /= max(1, stats.steps)
    stats.loss_i /= max(1, stats.steps)
    return stats

@torch.no_grad()
def validate(val_loader, model):
    model.eval()
    device = next(model.parameters()).device
    mae_total, count = 0.0, 0
    for images, gts in val_loader:
        images = images.to(device, non_blocking=True)
        gts    = gts.to(device, non_blocking=True)
        _, cam_im = model(images)
        cam_im = torch.sigmoid(cam_im)
        mae = eval_mae(cam_im, gts)
        mae_total += float(mae.cpu())
        count += 1
    return mae_total / max(1, count)

# ---------- lr schedule helper ----------
def adjust_lr(optimizer, epoch, decay_rate=0.1, decay_epoch=30):
    decay = decay_rate ** (epoch // decay_epoch)
    for pg in optimizer.param_groups:
        pg['lr'] = pg['initial_lr'] * decay  # requires setting initial_lr when building optimizer

# ---------- epoch driver ----------
def run_epoch(train_loader, val_loader, model, optimizer, epoch, opt, loss_func, best_mae, scaler=None):
    # adjust lr (optional: keep your existing scheme)
    adjust_lr(optimizer, epoch, opt.decay_rate, opt.decay_epoch)

    # train
    train_stats = train_one_epoch(train_loader, model, optimizer, epoch, opt, loss_func, scaler)

    # validate (if provided)
    val_mae = None
    if val_loader is not None:
        val_mae = validate(val_loader, model)
        print(f"[VAL] Epoch {epoch}  MAE={val_mae:.6f}")

    # save checkpoints
    os.makedirs(opt.save_model, exist_ok=True)
    if (epoch + 1) % opt.save_epoch == 0:
        torch.save(model.state_dict(), os.path.join(opt.save_model, f"SINet_{epoch+1}.pth"))

    if SAVE_BEST and val_mae is not None and val_mae < best_mae:
        best_mae = val_mae
        torch.save(model.state_dict(), os.path.join(opt.save_model, BEST_FILENAME))
        print(f"[CKPT] Saved best model (MAE {best_mae:.6f}) → {BEST_FILENAME}")

    # log summary
    print(f"[EPOCH {epoch}] Train: Loss_s={train_stats.loss_s:.4f} Loss_i={train_stats.loss_i:.4f} "
          f"{'(best so far)' if val_mae is not None and val_mae == best_mae else ''}")

    return best_mae

# ---------- convenient builder for optimizer with initial_lr ----------
def build_optimizer(model, lr=1e-4, weight_decay=1e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    for pg in opt.param_groups:
        pg.setdefault('initial_lr', lr)
    return opt

# ---------- AMP scaler ----------
def build_scaler():
    return torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [7]:
# cell 4: Search attention (safer: non-trainable kernel + device/dtype match)

import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import scipy.stats as st

def _get_kernel(kernlen=31, nsig=4):
    interval = (2*nsig+1.)/kernlen
    x = np.linspace(-nsig-interval/2., nsig+interval/2., kernlen+1)
    kern1d = np.diff(st.norm.cdf(x))
    kernel_raw = np.sqrt(np.outer(kern1d, kern1d))
    kernel = kernel_raw / kernel_raw.sum()
    return kernel.astype(np.float32)[None, None, ...]  # (1,1,K,K)

def min_max_norm(in_):
    max_ = in_.amax(dim=(2,3), keepdim=True)
    min_ = in_.amin(dim=(2,3), keepdim=True)
    return (in_ - min_) / (max_ - min_ + 1e-8)

class SA(nn.Module):
    """Holistic Search Attention: blur(attention) -> normalize -> gate features."""
    def __init__(self, ksize=31, nsig=4):
        super().__init__()
        k = torch.from_numpy(_get_kernel(ksize, nsig))      # (1,1,k,k), float32 CPU
        self.register_buffer('gaussian_kernel', k)          # non-trainable

    def forward(self, attention, x):
        # ensure kernel matches device/dtype
        k = self.gaussian_kernel.to(device=attention.device, dtype=attention.dtype)
        soft = F.conv2d(attention, k, padding=k.shape[-1]//2)
        soft = min_max_norm(soft)
        gate = torch.maximum(soft, attention)               # elementwise max
        return x * gate                                     # broadcast over channels


In [8]:
# cell 5 SINet (cleaned + PyTorch 2.x weights API)

import torch
import torch.nn as nn
import torchvision.models as tv_models

# expects:
# - ResNet_2Branch defined in Cell 1
# - SA defined in Cell 4 (recommended version with register_buffer)

class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1):
        super().__init__()
        self.conv = nn.Conv2d(in_planes, out_planes,
                              kernel_size=kernel_size, stride=stride,
                              padding=padding, dilation=dilation, bias=False)
        self.bn = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x

class RF(nn.Module):
    """Multi-branch receptive-field block (context aggregation)."""
    def __init__(self, in_channel, out_channel):
        super().__init__()
        self.relu = nn.ReLU(True)
        self.branch0 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
        )
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 3), padding=(0, 1)),
            BasicConv2d(out_channel, out_channel, kernel_size=(3, 1), padding=(1, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3),
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 5), padding=(0, 2)),
            BasicConv2d(out_channel, out_channel, kernel_size=(5, 1), padding=(2, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5),
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv2d(out_channel, out_channel, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7),
        )
        self.conv_cat = BasicConv2d(4 * out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x_cat = self.conv_cat(torch.cat((x0, x1, x2, x3), dim=1))
        x = self.relu(x_cat + self.conv_res(x))
        return x

class PDC_SM(nn.Module):
    """Partial Decoder Component for Search Module (coarse mask)."""
    def __init__(self, channel):
        super().__init__()
        self.relu = nn.ReLU(True)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample5 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)

        self.conv_concat2 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        self.conv_concat3 = BasicConv2d(4 * channel, 4 * channel, 3, padding=1)
        self.conv4 = BasicConv2d(4 * channel, 4 * channel, 3, padding=1)
        self.conv5 = nn.Conv2d(4 * channel, 1, 1)

    def forward(self, x1, x2, x3, x4):
        # x1: 32ch@44x44  (rf_low_sm)
        # x2: 32ch@44x44  (rf2_sm)
        # x3: 32ch@22x22  (rf3_sm)
        # x4: 32ch@11x11  (rf4_sm)
        x1_1 = x1
        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2
        x3_1 = self.conv_upsample2(self.upsample(self.upsample(x1))) * \
               self.conv_upsample3(self.upsample(x2)) * x3

        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)
        x2_2 = self.conv_concat2(x2_2)

        x3_2 = torch.cat((x3_1, self.conv_upsample5(self.upsample(x2_2)), x4), 1)
        x3_2 = self.conv_concat3(x3_2)

        x = self.conv4(x3_2)
        x = self.conv5(x)
        return x

class PDC_IM(nn.Module):
    """Partial Decoder Component for Identification Module (refined mask)."""
    def __init__(self, channel):
        super().__init__()
        self.relu = nn.ReLU(True)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample5 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)

        self.conv_concat2 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        self.conv_concat3 = BasicConv2d(3 * channel, 3 * channel, 3, padding=1)
        self.conv4 = BasicConv2d(3 * channel, 3 * channel, 3, padding=1)
        self.conv5 = nn.Conv2d(3 * channel, 1, 1)

    def forward(self, x1, x2, x3):
        # x1: 32ch@44x44 (rf2_im)
        # x2: 32ch@22x22 (rf3_im)
        # x3: 32ch@11x11 (rf4_im)
        x1_1 = x1
        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2
        x3_1 = self.conv_upsample2(self.upsample(self.upsample(x1))) * \
               self.conv_upsample3(self.upsample(x2)) * x3

        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)
        x2_2 = self.conv_concat2(x2_2)
        x3_2 = torch.cat((x3_1, self.conv_upsample5(self.upsample(x2_2))), 1)

        x3_2 = self.conv_concat3(x3_2)
        x = self.conv4(x3_2)
        x = self.conv5(x)
        return x

class SINet_ResNet50(nn.Module):
    """ResNet-2Branch encoder + RF blocks + PDC decoders + Search Attention."""
    def __init__(self, channel=32, opt=None):
        super().__init__()

        self.resnet = ResNet_2Branch()
        self.downSample = nn.MaxPool2d(2, stride=2)

        # --- Search Module (SM) ---
        self.rf_low_sm = RF(320, channel)      # concat(x0,x1) downsampled -> 320ch -> 32ch
        self.rf2_sm    = RF(3584, channel)     # cat(x2, up x3, up^2 x4) -> 3584 -> 32
        self.rf3_sm    = RF(3072, channel)     # cat(x3, up x4) -> 3072 -> 32
        self.rf4_sm    = RF(2048, channel)     # x4 -> 32
        self.pdc_sm    = PDC_SM(channel)

        # --- Identification Module (IM) ---
        self.rf2_im = RF(512, channel)         # x2_sa -> 32
        self.rf3_im = RF(1024, channel)        # x3_im -> 32
        self.rf4_im = RF(2048, channel)        # x4_im -> 32
        self.pdc_im = PDC_IM(channel)

        # utils
        self.upsample_2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.upsample_8 = nn.Upsample(scale_factor=8, mode='bilinear', align_corners=True)
        self.SA = SA()

        if self.training:
            self.initialize_weights()

    def forward(self, x):
        # ----- shared encoder -----
        x0 = self.resnet.conv1(x)              # (B, 64, 176,176) for 352x352 input before pool
        x0 = self.resnet.bn1(x0)
        x0 = self.resnet.relu(x0)
        x0 = self.resnet.maxpool(x0)           # (B, 64, 88,88)
        x1 = self.resnet.layer1(x0)            # (B, 256, 88,88)
        x2 = self.resnet.layer2(x1)            # (B, 512, 44,44)

        # ----- Stage-1: Search Module (SM) -----
        x01      = torch.cat((x0, x1), dim=1)       # (B, 64+256, 88,88) = 320
        x01_down = self.downSample(x01)             # (B, 320, 44,44)
        x01_sm   = self.rf_low_sm(x01_down)         # -> (B, 32, 44,44)

        x3_sm = self.resnet.layer3_1(x2)            # (B, 1024, 22,22)
        x4_sm = self.resnet.layer4_1(x3_sm)         # (B, 2048, 11,11)

        x2_sm_cat = torch.cat((
            x2,                                     # (512, 44,44)
            self.upsample_2(x3_sm),                 # (1024 -> 44,44)
            self.upsample_2(self.upsample_2(x4_sm)) # (2048 -> 44,44)
        ), dim=1)                                   # -> 3584 ch
        x3_sm_cat = torch.cat((x3_sm, self.upsample_2(x4_sm)), dim=1)  # 3072 ch

        x2_sm_rf = self.rf2_sm(x2_sm_cat)          # (B, 32, 44,44)
        x3_sm_rf = self.rf3_sm(x3_sm_cat)          # (B, 32, 22,22)
        x4_sm_rf = self.rf4_sm(x4_sm)              # (B, 32, 11,11)
        cam_sm   = self.pdc_sm(x4_sm_rf, x3_sm_rf, x2_sm_rf, x01_sm)  # (B,1,11,11)->upsampled later

        # ----- Switcher: Search Attention (SA) -----
        x2_sa = self.SA(cam_sm.sigmoid(), x2)      # gate x2 with coarse attention

        # ----- Stage-2: Identification Module (IM) -----
        x3_im   = self.resnet.layer3_2(x2_sa)      # (B, 1024, 22,22)
        x4_im   = self.resnet.layer4_2(x3_im)      # (B, 2048, 11,11)
        x2_imrf = self.rf2_im(x2_sa)               # (B, 32, 44,44)
        x3_imrf = self.rf3_im(x3_im)               # (B, 32, 22,22)
        x4_imrf = self.rf4_im(x4_im)               # (B, 32, 11,11)
        cam_im  = self.pdc_im(x4_imrf, x3_imrf, x2_imrf)

        # ----- outputs (to input size) -----
        return self.upsample_8(cam_sm), self.upsample_8(cam_im)

    def initialize_weights(self):
        """Load ImageNet weights into the custom 2-branch ResNet backbone."""
        # PyTorch 2.x API (fallback to old if needed)
        try:
            from torchvision.models import resnet50, ResNet50_Weights
            resnet50_pre = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        except Exception:
            resnet50_pre = tv_models.resnet50(pretrained=True)

        pretrained_dict = resnet50_pre.state_dict()
        all_params = {}
        # map shared + two-branch layers by removing suffixes _1 / _2
        for k, v in self.resnet.state_dict().items():
            if k in pretrained_dict:
                all_params[k] = pretrained_dict[k]
            elif '_1' in k:
                name = k.replace('_1', '')
                if name in pretrained_dict:
                    all_params[k] = pretrained_dict[name]
            elif '_2' in k:
                name = k.replace('_2', '')
                if name in pretrained_dict:
                    all_params[k] = pretrained_dict[name]

        assert len(all_params) == len(self.resnet.state_dict()), \
            f"Pretrained mapping mismatch: got {len(all_params)}/{len(self.resnet.state_dict())}"

        self.resnet.load_state_dict(all_params)
        print('[INFO] initialize weights from ImageNet ResNet50')


In [ ]:
# cell 6: launch training (clean + val split + save-best + AMP + optional CAMO-train)

import os
import torch
import argparse
from torch.utils.data import DataLoader, ConcatDataset, random_split

# toggles
USE_CAMO_TRAIN = False   # set True if you also have CAMO train paths below
VAL_RATIO      = 0.10    # 10% of training data for validation
NUM_WORKERS    = 4       # Kaggle T4 safe; bump to 8/12 if stable

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--epoch',      type=int,   default=30)
    parser.add_argument('--lr',         type=float, default=1e-4)
    parser.add_argument('--batchsize',  type=int,   default=12)   # safer default than 36
    parser.add_argument('--trainsize',  type=int,   default=352)
    parser.add_argument('--clip',       type=float, default=0.5)
    parser.add_argument('--decay_rate', type=float, default=0.1)
    parser.add_argument('--decay_epoch',type=int,   default=30)
    parser.add_argument('--gpu',        type=int,   default=0)
    parser.add_argument('--save_epoch', type=int,   default=10)
    parser.add_argument('--save_model', type=str,   default='/kaggle/working/Models/SINet/')
    # COD10K (required)
    parser.add_argument('--train_img_dir', type=str, default='/kaggle/input/cod10k-dataset/COD10K-v3/Train/Image/')
    parser.add_argument('--train_gt_dir',  type=str, default='/kaggle/input/cod10k-dataset/COD10K-v3/Train/GT_Object/')
    # CAMO train (optional, used only if USE_CAMO_TRAIN=True and paths exist)
    parser.add_argument('--camo_train_img_dir', type=str, default='/kaggle/input/camo-dataset/CAMO-V.1.0-CVIU2019/Images/Train/')
    parser.add_argument('--camo_train_gt_dir',  type=str, default='/kaggle/input/camo-dataset/CAMO-V.1.0-CVIU2019/GT/Train/')
    opt, unknown = parser.parse_known_args()

    torch.cuda.set_device(opt.gpu)
    torch.backends.cudnn.benchmark = True

    # ----- build model -----
    model_SINet = SINet_ResNet50(channel=32).cuda()
    print('-' * 30, "Initialized SINet_ResNet50", '-' * 30)

    # ----- optimizer / loss / AMP (from trainer cell 3 modernized) -----
    optimizer  = build_optimizer(model_SINet, lr=opt.lr, weight_decay=1e-4)
    loss_func  = build_loss()
    scaler     = build_scaler()

    # ----- datasets -----
    # from your corrected dataloader cell: CamObjDataset, get_loader, USE_AUG toggled there
    cod_ds  = CamObjDataset(opt.train_img_dir,  opt.train_gt_dir,  trainsize=opt.trainsize, use_aug=True)

    if USE_CAMO_TRAIN and os.path.isdir(opt.camo_train_img_dir) and os.path.isdir(opt.camo_train_gt_dir):
        camo_ds = CamObjDataset(opt.camo_train_img_dir, opt.camo_train_gt_dir, trainsize=opt.trainsize, use_aug=True)
        full_train = ConcatDataset([cod_ds, camo_ds])
        print(f"[INFO] Using COD10K + CAMO-Train  → total samples: {len(cod_ds)} + {len(camo_ds)} = {len(cod_ds)+len(camo_ds)}")
    else:
        full_train = cod_ds
        print(f"[INFO] Using COD10K only → samples: {len(cod_ds)}")

    # ----- train/val split -----
    n_total = len(full_train)
    n_val   = max(1, int(VAL_RATIO * n_total))
    n_train = n_total - n_val
    g = torch.Generator().manual_seed(42)  # reproducible split
    train_ds, val_ds = random_split(full_train, [n_train, n_val], generator=g)

    train_loader = DataLoader(train_ds, batch_size=opt.batchsize, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
    # turn off aug for val: wrap CAMO class with use_aug=False if needed
    if isinstance(val_ds.dataset, ConcatDataset):
        # rebuild a no-aug validation dataset for each constituent
        parts = []
        for d in val_ds.dataset.datasets:
            parts.append(CamObjDataset(getattr(d, 'images')[0].rsplit('/',2)[0]+'/',   # not perfect; simplest is rebuild explicitly
                                       getattr(d, 'gts')[0].rsplit('/',2)[0]+'/',
                                       trainsize=opt.trainsize, use_aug=False))
        val_base = ConcatDataset(parts)
        # keep only the indices selected by random_split
        val_loader = DataLoader(torch.utils.data.Subset(val_base, val_ds.indices),
                                batch_size=opt.batchsize, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
    else:
        # simpler: build a clean no-aug dataset and subset with the split indices
        val_clean = CamObjDataset(opt.train_img_dir, opt.train_gt_dir, trainsize=opt.trainsize, use_aug=False)
        val_loader = DataLoader(torch.utils.data.Subset(val_clean, val_ds.indices),
                                batch_size=opt.batchsize, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

    print(f"[INFO] Train/Val sizes → {len(train_ds)} / {len(val_ds)} | Batch={opt.batchsize} | LR={opt.lr}")
    os.makedirs(opt.save_model, exist_ok=True)

    # ----- training loop -----
    best_mae = float('inf')
    for epoch_iter in range(0, opt.epoch):
        best_mae = run_epoch(train_loader, val_loader, model_SINet, optimizer,
                             epoch_iter, opt, loss_func, best_mae, scaler)


In [16]:
# cell 7: launch testing (clean + no_grad + interpolate + best ckpt)

import os
import argparse
import torch
import torch.nn.functional as F
import numpy as np
import imageio

# expects:
# - SINet_ResNet50 defined (Cell 5)
# - TestDataset defined (Cell 2 corrected)

USE_AMP = True  # set False if you hit issues on CPU-only

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--testsize',  type=int,   default=352, help='network input size')
    parser.add_argument('--model_path',type=str,   default='/kaggle/input/sinet-best/pytorch/default/1/SINet_best (10).pth')
    parser.add_argument('--test_img_dir', type=str, default='/kaggle/input/cod10k-dataset/COD10K-v3/Test/Image/')
    parser.add_argument('--test_gt_dir',  type=str, default='/kaggle/input/cod10k-dataset/COD10K-v3/Test/GT_Object/')
    parser.add_argument('--test_save',    type=str, default='/kaggle/working/Results/SINet/COD10K/')
    parser.add_argument('--gpu',       type=int,   default=0)
    opt, _ = parser.parse_known_args()

    # device
    device = torch.device(f'cuda:{opt.gpu}' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda':
        torch.cuda.set_device(opt.gpu)
        torch.backends.cudnn.benchmark = True

    # model
    model = SINet_ResNet50().to(device)
    state = torch.load(opt.model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()

    # data
    os.makedirs(opt.test_save, exist_ok=True)
    test_loader = TestDataset(image_root=opt.test_img_dir, gt_root=opt.test_gt_dir, testsize=opt.testsize)

    print(f"[INFO] Testing on: {opt.test_img_dir}")
    print(f"[INFO] Saving to  : {opt.test_save}")

    # loop
    from tqdm import trange
    n = test_loader.size
    img_count = 1

    with torch.no_grad():
        for _ in trange(n, desc="Inference"):
            image, gt, name = test_loader.load_data()  # image: (1,3,H,W) tensor; gt: (1,1,H,W) tensor
            H, W = gt.shape[-2], gt.shape[-1]

            image = image.to(device, non_blocking=True)

            if USE_AMP and device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    _, cam = model(image)
            else:
                _, cam = model(image)

            # upsample to GT size, sigmoid to [0,1], squeeze to HxW
            cam = torch.sigmoid(F.interpolate(cam, size=(H, W), mode='bilinear', align_corners=False))
            cam = cam[0, 0].detach().cpu().numpy()
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

            # save
            save_path = os.path.join(opt.test_save, name)  # name already ends with .png in TestDataset
            imageio.imwrite(save_path, (cam * 255).astype(np.uint8))

            # log
            print(f"[Eval-Test] Image: {name} ({img_count}/{n})  →  {save_path}")
            img_count += 1

    print("\n[Congratulations! Testing Done]")


[INFO] initialize weights from ImageNet ResNet50
[INFO] Testing on: /kaggle/input/cod10k-dataset/COD10K-v3/Test/Image/
[INFO] Saving to  : /kaggle/working/Results/SINet/COD10K/


Inference:   0%|          | 0/4000 [00:00<?, ?it/s]/tmp/ipykernel_37/2985554538.py:58: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Inference:   0%|          | 2/4000 [00:00<03:33, 18.69it/s]

[Eval-Test] Image: COD10K-CAM-1-Aquatic-1-BatFish-2.png (1/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-CAM-1-Aquatic-1-BatFish-2.png
[Eval-Test] Image: COD10K-CAM-1-Aquatic-1-BatFish-4.png (2/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-CAM-1-Aquatic-1-BatFish-4.png
[Eval-Test] Image: COD10K-CAM-1-Aquatic-1-BatFish-5.png (3/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-CAM-1-Aquatic-1-BatFish-5.png


Inference:   0%|          | 4/4000 [00:00<03:29, 19.09it/s]

[Eval-Test] Image: COD10K-CAM-1-Aquatic-1-BatFish-6.png (4/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-CAM-1-Aquatic-1-BatFish-6.png


...

[Eval-Test] Image: COD10K-NonCAM-5-Background-5-Vegetation-4906.png (3991/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-NonCAM-5-Background-5-Vegetation-4906.png
[Eval-Test] Image: COD10K-NonCAM-5-Background-5-Vegetation-4909.png (3992/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-NonCAM-5-Background-5-Vegetation-4909.png
[Eval-Test] Image: COD10K-NonCAM-5-Background-5-Vegetation-4912.png (3993/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-NonCAM-5-Background-5-Vegetation-4912.png
[Eval-Test] Image: COD10K-NonCAM-5-Background-5-Vegetation-4915.png (3994/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-NonCAM-5-Background-5-Vegetation-4915.png


Inference: 100%|█████████▉| 3998/4000 [03:19<00:00, 17.69it/s]

[Eval-Test] Image: COD10K-NonCAM-5-Background-5-Vegetation-4916.png (3995/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-NonCAM-5-Background-5-Vegetation-4916.png
[Eval-Test] Image: COD10K-NonCAM-5-Background-5-Vegetation-4918.png (3996/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-NonCAM-5-Background-5-Vegetation-4918.png
[Eval-Test] Image: COD10K-NonCAM-5-Background-5-Vegetation-4920.png (3997/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-NonCAM-5-Background-5-Vegetation-4920.png
[Eval-Test] Image: COD10K-NonCAM-5-Background-5-Vegetation-4923.png (3998/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-NonCAM-5-Background-5-Vegetation-4923.png


Inference: 100%|██████████| 4000/4000 [03:19<00:00, 20.01it/s]

[Eval-Test] Image: COD10K-NonCAM-5-Background-5-Vegetation-4929.png (3999/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-NonCAM-5-Background-5-Vegetation-4929.png
[Eval-Test] Image: COD10K-NonCAM-5-Background-5-Vegetation-4931.png (4000/4000)  →  /kaggle/working/Results/SINet/COD10K/COD10K-NonCAM-5-Background-5-Vegetation-4931.png

[Congratulations! Testing Done]


In [ ]:
# Cell 8 (robust viewer with diagnostics for COD10K)

import os
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

PRED_DIR  = '/kaggle/working/Results/SINet/COD10K/'   # <-- from Cell 7
IMAGE_DIR = '/kaggle/input/cod10k-dataset/COD10K-v3/Test/Image/'            # GT images

# --- quick diagnostics ---
print("[DIAG] pred dir exists:", os.path.isdir(PRED_DIR), "→", PRED_DIR)
print("[DIAG] image dir exists:", os.path.isdir(IMAGE_DIR), "→", IMAGE_DIR)

pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.lower().endswith('.png')])
img_files  = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(IMG_EXTS)])

print("[DIAG] #preds:", len(pred_files))
print("[DIAG] #images:", len(img_files))

if len(img_files) == 0:
    raise RuntimeError("No images found in IMAGE_DIR. Check the path.")

def _to_numpy_gray(pil_img):
    a = np.array(pil_img)
    if a.ndim == 3:  # RGB -> gray if needed
        a = a[..., 0]
    return a

def test_visualize(prediction_root_path,
                   image_root_path,
                   start=0,
                   end=20,
                   overlay=False,
                   overlay_alpha=0.5):
    img_names = sorted([f for f in os.listdir(image_root_path) if f.lower().endswith(IMG_EXTS)])
    img_names = img_names[start:end]
    if not img_names:
        print(f"[WARN] No image names in the requested slice [{start}:{end}]. "
              f"Total images available: {len(os.listdir(image_root_path))}.")
        return

    for name in img_names:
        stem = Path(name).stem
        img_path  = os.path.join(image_root_path, name)
        pred_path = os.path.join(prediction_root_path, stem + ".png")  # Cell 7 saves .png

        # read inputs
        img = Image.open(img_path).convert("RGB")

        if not os.path.exists(pred_path):
            print(f"[MISS] Prediction not found for {stem} → {pred_path}")
            # still show the image, skip pred
            plt.figure(figsize=(6, 5))
            plt.imshow(np.array(img)); plt.title(f"Image (missing pred: {stem})"); plt.axis("off")
            plt.show()
            continue

        pred = Image.open(pred_path).convert("L")
        pred_np = _to_numpy_gray(pred)

        # show
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1); plt.imshow(np.array(img)); plt.title("Image"); plt.axis("off")
        if overlay:
            import matplotlib.cm as cm
            plt.subplot(1, 2, 2)
            plt.imshow(np.array(img))
            plt.imshow(pred_np, cmap='jet', alpha=overlay_alpha)
            plt.title("Prediction (overlay)"); plt.axis("off")
        else:
            plt.subplot(1, 2, 2); plt.imshow(pred_np, cmap='gray'); plt.title("Prediction"); plt.axis("off")
        plt.tight_layout(); plt.show()

# ---- call it (adjust the slice to something that exists) ----
test_visualize(
    prediction_root_path=PRED_DIR,
    image_root_path=IMAGE_DIR,
    start=0,   # try 0–20 first to ensure files exist in that range
    end=20,
    overlay=False
)

In [ ]:
# Permanently save prediction results in Kaggle Notebook Version

import shutil
import os

src = "/kaggle/working/Results/SINet/COD10K/"   # where your predictions are now
dst = "/kaggle/output/COD10K_Predictions/"     # this is permanent storage

# Create output folder if missing
os.makedirs(dst, exist_ok=True)

# Copy everything from working → output
shutil.copytree(src, dst, dirs_exist_ok=True)

print("Predictions copied to:", dst)
print("Now go to: File → Save Version → select 'Save output'")

In [ ]:
# Cell 10: Lightweight Offline Evaluation (memory-safe, no extra libs) change karna he

import os, gc
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

def _read_gray(p):
    return cv2.imread(p, cv2.IMREAD_GRAYSCALE)

def _to_uint8(arr):
    if arr is None:
        return None
    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        if arr.max() <= 1.0:
            arr *= 255.0
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    return arr

def evaluate(mask_root, pred_root, limit=None, compute_pr=False):
    """
    limit: evaluate first N images (for a quick sanity run). None = full set.
    compute_pr: if True, accumulates a 256-threshold PR curve (slightly heavier).
    """
    mask_names = sorted(os.listdir(mask_root))
    if limit is not None:
        mask_names = mask_names[:limit]

    mae_sum = 0.0
    n_count = 0
    iou_sum = 0.0
    dice_sum = 0.0

    # Optional PR curve (256 thresholds)
    if compute_pr:
        P_sum = np.zeros(256, dtype=np.float64)
        R_sum = np.zeros(256, dtype=np.float64)
        pr_count = 0
        beta2 = 0.3

    for name in tqdm(mask_names, desc="Evaluating"):
        stem = Path(name).stem
        gt_path   = os.path.join(mask_root, name)
        pred_path = os.path.join(pred_root, stem + ".png")

        gt   = _read_gray(gt_path)
        pred = _read_gray(pred_path)
        if gt is None or pred is None:
            # skip if either missing
            continue

        # binarize GT to {0,1}
        gt = (gt >= 128).astype(np.uint8)

        # ensure same size
        if gt.shape != pred.shape:
            pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_LINEAR)

        pred = _to_uint8(pred)

        # ---- MAE (pred treated as [0,1]) ----
        mae_sum += float(np.mean(np.abs((pred.astype(np.float32) / 255.0) - gt)))
        n_count += 1

        # ---- Adaptive threshold (Fm-style) ----
        T = int(min(255, max(0, round(2 * float(pred.mean())))))
        pred_bin = (pred >= T)

        tp = np.logical_and(pred_bin, gt == 1).sum()
        fp = np.logical_and(pred_bin, gt == 0).sum()
        fn = np.logical_and(~pred_bin, gt == 1).sum()

        iou = tp / (tp + fp + fn + 1e-9)
        dice = (2 * tp) / (2 * tp + fp + fn + 1e-9)
        iou_sum += iou
        dice_sum += dice

        # ---- Optional PR curve accumulation (slightly heavier) ----
        if compute_pr:
            pr = pred.ravel()
            gg = gt.ravel()
            pos = (gg == 1)
            posN = int(pos.sum())
            if posN > 0:
                # histograms (uint32 to stay safe)
                hist_all = np.bincount(pr, minlength=256).astype(np.float64)
                hist_pos = np.bincount(pr[pos], minlength=256).astype(np.float64)

                c_all = np.cumsum(hist_all[::-1])[::-1]
                c_pos = np.cumsum(hist_pos[::-1])[::-1]
                tp_curve = c_pos
                fp_curve = c_all - c_pos
                fn_curve = posN - tp_curve

                P = tp_curve / (tp_curve + fp_curve + 1e-9)
                R = tp_curve / (tp_curve + fn_curve + 1e-9)
                P_sum += P
                R_sum += R
                pr_count += 1

        # free per-iteration memory
        del gt, pred, pred_bin
        gc.collect()

    if n_count == 0:
        print("[ERR] No valid (GT, pred) pairs found. Check paths & filenames.")
        return None

    results = {
        "MAE": mae_sum / n_count,
        "IoU@Adaptive": iou_sum / n_count,
        "Dice@Adaptive": dice_sum / n_count,
        "N_evaluated": int(n_count),
    }

    if compute_pr:
        P = P_sum / (pr_count + 1e-9)
        R = R_sum / (pr_count + 1e-9)
        beta2 = 0.3
        F = (1 + beta2) * P * R / (beta2 * P + R + 1e-9)
        results.update({
            "maxF(β²=0.3)": float(F.max()),
            "meanF(β²=0.3)": float(F.mean()),
        })

    return results

# --- set your paths ---
mask_root = "/kaggle/input/cod10k-dataset/COD10K-v3/Test/GT_Object/"
pred_root = "/kaggle/working/Results/SINet/COD10K/"

# First run a tiny sanity subset to ensure it doesn't snap:
print("Quick sanity pass on 50 images...")
res_small = evaluate(mask_root, pred_root, limit=50, compute_pr=False)
print(res_small)

# Then full run (toggle compute_pr to True if you want PR-based F-measures):
print("\nFull evaluation (no PR curve):")
res_full = evaluate(mask_root, pred_root, limit=None, compute_pr=False)
print(res_full)

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

def evaluate_iou_dice(mask_root, pred_root, threshold=128):
    mask_names = sorted(os.listdir(mask_root))
    iou_list, dice_list, acc_list = [], [], []

    for name in tqdm(mask_names):
        stem = name.rsplit('.', 1)[0]
        mask = cv2.imread(os.path.join(mask_root, name), cv2.IMREAD_GRAYSCALE)
        pred = cv2.imread(os.path.join(pred_root, stem + ".png"), cv2.IMREAD_GRAYSCALE)

        if pred is None or mask is None:
            continue

        # resize pred if needed
        if mask.shape != pred.shape:
            pred = cv2.resize(pred, (mask.shape[1], mask.shape[0]))

        # binarize
        mask_bin = (mask >= threshold).astype(np.uint8)
        pred_bin = (pred >= threshold).astype(np.uint8)

        # pixel accuracy
        acc = (mask_bin == pred_bin).mean()
        acc_list.append(acc)

        # IoU
        inter = np.logical_and(mask_bin, pred_bin).sum()
        union = np.logical_or(mask_bin, pred_bin).sum()
        iou = inter / (union + 1e-7)
        iou_list.append(iou)

        # Dice
        dice = (2 * inter) / (mask_bin.sum() + pred_bin.sum() + 1e-7)
        dice_list.append(dice)

    return {
        "Pixel_Accuracy (bad metric)": float(np.mean(acc_list)),
        "IoU (Jaccard)": float(np.mean(iou_list)),
        "Dice / F1 Score": float(np.mean(dice_list)),
    }

# Run it
mask_root = '/kaggle/input/cod10k-dataset/COD10K-v3/Test/GT_Object/'
pred_root = '/kaggle/working/Results/SINet/COD10K/'
res = evaluate_iou_dice(mask_root, pred_root)
print(res)


In [14]:
# cell 10: launch testing for Military Personnel Data (clean + no_grad + interpolate + AMP)

import os
import argparse
import torch
import torch.nn.functional as F
import numpy as np
import imageio

# expects:
# - SINet_ResNet50 (Cell 5)
# - TestDataset (Cell 2 corrected)

USE_AMP = True

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--testsize',     type=int, default=352, help='network input size')
    parser.add_argument('--model_path',   type=str, default='/kaggle/input/sinet-best/pytorch/default/1/SINet_best (10).pth')
    parser.add_argument('--test_img_dir', type=str, default='/kaggle/input/military-personnel-dataset-dataset/CamouflageData/img/')
    parser.add_argument('--test_gt_dir',  type=str, default='/kaggle/input/military-personnel-dataset-dataset/CamouflageData/gt/')
    parser.add_argument('--test_save',    type=str, default='/kaggle/working/Results/SINet/Military/')
    parser.add_argument('--gpu',          type=int, default=0)
    opt, _ = parser.parse_known_args()

    # device
    device = torch.device(f'cuda:{opt.gpu}' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda':
        torch.cuda.set_device(opt.gpu)
        torch.backends.cudnn.benchmark = True

    # model
    model = SINet_ResNet50().to(device)
    state = torch.load(opt.model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()

    # data
    os.makedirs(opt.test_save, exist_ok=True)
    test_loader = TestDataset(image_root=opt.test_img_dir, gt_root=opt.test_gt_dir, testsize=opt.testsize)
    print(f"[INFO] Testing on: {opt.test_img_dir}")
    print(f"[INFO] Saving to  : {opt.test_save}")

    from tqdm import trange
    n = test_loader.size
    img_count = 1

    with torch.no_grad():
        for _ in trange(n, desc="Inference (MPD)"):
            image, gt, name = test_loader.load_data()  # image: (1,3,h,w); gt: (1,1,H,W) PIL->tensor (for size)
            H, W = gt.shape[-2], gt.shape[-1]
            image = image.to(device, non_blocking=True)

            if USE_AMP and device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    _, cam = model(image)
            else:
                _, cam = model(image)

            cam = torch.sigmoid(F.interpolate(cam, size=(H, W), mode='bilinear', align_corners=False))
            cam = cam[0, 0].detach().cpu().numpy()
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

            save_path = os.path.join(opt.test_save, name)   # name already .png in TestDataset
            imageio.imwrite(save_path, (cam * 255).astype(np.uint8))
            print(f"[Eval-Test MPD] Image: {name} ({img_count}/{n}) → {save_path}")
            img_count += 1

    print("\n[Congratulations! MPD Testing Done]")


[INFO] initialize weights from ImageNet ResNet50
[INFO] Testing on: /kaggle/input/military-personnel-dataset-dataset/CamouflageData/img/
[INFO] Saving to  : /kaggle/working/Results/SINet/Military/


Inference (MPD):   0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_37/375240733.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Inference (MPD):   0%|          | 3/1000 [00:00<00:45, 22.14it/s]

[Eval-Test MPD] Image: dataset01_01_00002256.png (1/1000) → /kaggle/working/Results/SINet/Military/dataset01_01_00002256.png
[Eval-Test MPD] Image: dataset01_01_00002277.png (2/1000) → /kaggle/working/Results/SINet/Military/dataset01_01_00002277.png
[Eval-Test MPD] Image: dataset01_01_00002310.png (3/1000) → /kaggle/working/Results/SINet/Military/dataset01_01_00002310.png
[Eval-Test MPD] Image: dataset01_01_00002586.png (4/1000) → /kaggle/working/Results/SINet/Military/dataset01_01_00002586.png
[Eval-Test MPD] Image: dataset01_01_00002643.png (5/1000) → /kaggle/working/Results/SINet/Military/dataset01_01_00002643.png


Inference (MPD):   1%|          | 9/1000 [00:00<00:41, 24.15it/s]

[Eval-Test MPD] Image: dataset01_02_00004128.png (6/1000) → /kaggle/working/Results/SINet/Military/dataset01_02_00004128.png
[Eval-Test MPD] Image: dataset01_03_00005229.png (7/1000) → /kaggle/working/Results/SINet/Military/dataset01_03_00005229.png
[Eval-Test MPD] Image: dataset01_03_00005505.png (8/1000) → /kaggle/working/Results/SINet/Military/dataset01_03_00005505.png
[Eval-Test MPD] Image: dataset01_03_00005853.png (9/1000) → /kaggle/working/Results/SINet/Military/dataset01_03_00005853.png
[Eval-Test MPD] Image: dataset01_03_00005859.png (10/1000) → /kaggle/working/Results/SINet/Military/dataset01_03_00005859.png


...

[Eval-Test MPD] Image: dataset20_14_00018453.png (988/1000) → /kaggle/working/Results/SINet/Military/dataset20_14_00018453.png
[Eval-Test MPD] Image: dataset20_14_00018672.png (989/1000) → /kaggle/working/Results/SINet/Military/dataset20_14_00018672.png
[Eval-Test MPD] Image: dataset20_15_00016104.png (990/1000) → /kaggle/working/Results/SINet/Military/dataset20_15_00016104.png
[Eval-Test MPD] Image: dataset20_15_00016233.png (991/1000) → /kaggle/working/Results/SINet/Military/dataset20_15_00016233.png
[Eval-Test MPD] Image: dataset20_15_00016245.png (992/1000) → /kaggle/working/Results/SINet/Military/dataset20_15_00016245.png


Inference (MPD): 100%|█████████▉| 996/1000 [00:41<00:00, 23.52it/s]

[Eval-Test MPD] Image: dataset20_15_00020004.png (993/1000) → /kaggle/working/Results/SINet/Military/dataset20_15_00020004.png
[Eval-Test MPD] Image: dataset20_16_00016752.png (994/1000) → /kaggle/working/Results/SINet/Military/dataset20_16_00016752.png
[Eval-Test MPD] Image: dataset20_16_00016872.png (995/1000) → /kaggle/working/Results/SINet/Military/dataset20_16_00016872.png
[Eval-Test MPD] Image: dataset20_16_00021228.png (996/1000) → /kaggle/working/Results/SINet/Military/dataset20_16_00021228.png
[Eval-Test MPD] Image: dataset20_16_00021555.png (997/1000) → /kaggle/working/Results/SINet/Military/dataset20_16_00021555.png


Inference (MPD): 100%|██████████| 1000/1000 [00:41<00:00, 24.18it/s]

[Eval-Test MPD] Image: dataset20_17_00022194.png (998/1000) → /kaggle/working/Results/SINet/Military/dataset20_17_00022194.png
[Eval-Test MPD] Image: dataset20_17_00022665.png (999/1000) → /kaggle/working/Results/SINet/Military/dataset20_17_00022665.png
[Eval-Test MPD] Image: dataset20_17_00022710.png (1000/1000) → /kaggle/working/Results/SINet/Military/dataset20_17_00022710.png

[Congratulations! MPD Testing Done]


In [ ]:
# Cell 8 (robust viewer with diagnostics for COD10K)

import os
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

PRED_DIR  = '/kaggle/working/Results/SINet/Military'   # <-- from Cell 7
IMAGE_DIR = '/kaggle/input/military-personnel-dataset-dataset/CamouflageData/img'           

# --- quick diagnostics ---
print("[DIAG] pred dir exists:", os.path.isdir(PRED_DIR), "→", PRED_DIR)
print("[DIAG] image dir exists:", os.path.isdir(IMAGE_DIR), "→", IMAGE_DIR)

pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.lower().endswith('.png')])
img_files  = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(IMG_EXTS)])

print("[DIAG] #preds:", len(pred_files))
print("[DIAG] #images:", len(img_files))

if len(img_files) == 0:
    raise RuntimeError("No images found in IMAGE_DIR. Check the path.")

def _to_numpy_gray(pil_img):
    a = np.array(pil_img)
    if a.ndim == 3:  # RGB -> gray if needed
        a = a[..., 0]
    return a

def test_visualize(prediction_root_path,
                   image_root_path,
                   start=0,
                   end=20,
                   overlay=False,
                   overlay_alpha=0.5):
    img_names = sorted([f for f in os.listdir(image_root_path) if f.lower().endswith(IMG_EXTS)])
    img_names = img_names[start:end]
    if not img_names:
        print(f"[WARN] No image names in the requested slice [{start}:{end}]. "
              f"Total images available: {len(os.listdir(image_root_path))}.")
        return

    for name in img_names:
        stem = Path(name).stem
        img_path  = os.path.join(image_root_path, name)
        pred_path = os.path.join(prediction_root_path, stem + ".png")  # Cell 7 saves .png

        # read inputs
        img = Image.open(img_path).convert("RGB")

        if not os.path.exists(pred_path):
            print(f"[MISS] Prediction not found for {stem} → {pred_path}")
            # still show the image, skip pred
            plt.figure(figsize=(6, 5))
            plt.imshow(np.array(img)); plt.title(f"Image (missing pred: {stem})"); plt.axis("off")
            plt.show()
            continue

        pred = Image.open(pred_path).convert("L")
        pred_np = _to_numpy_gray(pred)

        # show
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1); plt.imshow(np.array(img)); plt.title("Image"); plt.axis("off")
        if overlay:
            import matplotlib.cm as cm
            plt.subplot(1, 2, 2)
            plt.imshow(np.array(img))
            plt.imshow(pred_np, cmap='jet', alpha=overlay_alpha)
            plt.title("Prediction (overlay)"); plt.axis("off")
        else:
            plt.subplot(1, 2, 2); plt.imshow(pred_np, cmap='gray'); plt.title("Prediction"); plt.axis("off")
        plt.tight_layout(); plt.show()

# ---- call it (adjust the slice to something that exists) ----
test_visualize(
    prediction_root_path=PRED_DIR,
    image_root_path=IMAGE_DIR,
    start=0,   # try 0–20 first to ensure files exist in that range
    end=20,
    overlay=False
)

In [ ]:
import shutil

pred_folder = "/kaggle/working/Results/SINet/Military/"
zip_save_path = "/kaggle/working/military_results.zip"

# Create zip
shutil.make_archive(zip_save_path.replace(".zip", ""), 'zip', pred_folder)

print(f" Zipped successfully!\nSaved to: {zip_save_path}")
print(" Go to Kaggle right sidebar → Files → download `military_results.zip`")


In [ ]:
# cell 11: evaluate MPD predictions (S-measure, wF-measure, MAE)

import os
import cv2
from tqdm import tqdm
import numpy as np

# pip install py_sod_metrics  (if not already)
from py_sod_metrics import MAE, Emeasure, Fmeasure, Smeasure, WeightedFmeasure

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp')

def evaluate_dataset(mask_root, pred_root, desc="Scoring"):
    mask_names = sorted([f for f in os.listdir(mask_root) if f.lower().endswith(IMG_EXTS)])
    if len(mask_names) == 0:
        print(f"[WARN] No GT files in {mask_root}")
        return None

    WFM = WeightedFmeasure(0.3)  # popular for COD/SOD
    SM  = Smeasure()
    M   = MAE()
    # Optional extras:
    # EM = Emeasure()
    # FM = Fmeasure()

    missing = 0
    for name in tqdm(mask_names, total=len(mask_names), desc=desc):
        stem = os.path.splitext(name)[0]
        mask_path = os.path.join(mask_root, name)
        pred_path = os.path.join(pred_root, stem + ".png")  # we saved preds as .png

        if not os.path.exists(pred_path):
            missing += 1
            continue

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
        if mask is None or pred is None:
            continue

        # resize pred to GT size if needed
        if mask.shape != pred.shape:
            pred = cv2.resize(pred, (mask.shape[1], mask.shape[0]), interpolation=cv2.INTER_LINEAR)

        # ensure uint8 [0,255]
        if pred.dtype != np.uint8:
            pred = pred.astype(np.float32)
            if pred.max() <= 1.0:
                pred = (pred * 255.0).round()
            pred = np.clip(pred, 0, 255).astype(np.uint8)
        if mask.dtype != np.uint8:
            mask = np.clip(mask, 0, 255).astype(np.uint8)

        # step metrics
        WFM.step(pred=pred, gt=mask)
        SM.step(pred=pred, gt=mask)
        M.step(pred=pred, gt=mask)
        # EM.step(pred=pred, gt=mask)
        # FM.step(pred=pred, gt=mask)

    results = {
        "Smeasure":  SM.get_results().get("sm", np.nan),
        "wFmeasure": WFM.get_results().get("wfm", np.nan),
        "MAE":       M.get_results().get("mae", np.nan),
        # "adpEm":   EM.get_results()["em"]["adp"],
        # "meanEm":  EM.get_results()["em"]["curve"].mean(),
        # "maxEm":   EM.get_results()["em"]["curve"].max(),
        # "adpFm":   FM.get_results()["fm"]["adp"],
        # "meanFm":  FM.get_results()["fm"]["curve"].mean(),
        # "maxFm":   FM.get_results()["fm"]["curve"].max(),
    }
    if missing:
        print(f"[INFO] Missing preds for {missing}/{len(mask_names)} images (skipped).")
    return results

# ---- run on MPD ----
gt_root   = '/kaggle/input/military-personnel-dataset-dataset/CamouflageData/gt/'
pred_root = '/kaggle/working/Results/SINet/Military/'  # match your Cell 10 save path

res = evaluate_dataset(gt_root, pred_root, desc="Scoring MPD")
print(res if res is not None else "[ERR] Evaluation failed.")


In [ ]:
# cell 12: evaluate MPD predictions (robust paths + stem mapping + dtype safety)

import os
import cv2
from tqdm import tqdm
import numpy as np

# pip install py_sod_metrics
from py_sod_metrics import MAE, Smeasure, WeightedFmeasure  # add EM/FM if you want

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp')

def evaluate_dataset(mask_root, pred_root, desc="Scoring"):
    mask_names = sorted([f for f in os.listdir(mask_root) if f.lower().endswith(IMG_EXTS)])
    if len(mask_names) == 0:
        print(f"[WARN] No GT files in {mask_root}")
        return None

    WFM = WeightedFmeasure(0.3)
    SM  = Smeasure()
    M   = MAE()

    missing = 0
    for name in tqdm(mask_names, total=len(mask_names), desc=desc):
        stem = os.path.splitext(name)[0]
        mask_path = os.path.join(mask_root, name)
        # preds saved as PNG with same stem
        pred_path = os.path.join(pred_root, stem + ".png")

        if not os.path.exists(pred_path):
            missing += 1
            continue

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
        if mask is None or pred is None:
            continue

        # resize pred to GT if needed
        if mask.shape != pred.shape:
            pred = cv2.resize(pred, (mask.shape[1], mask.shape[0]), interpolation=cv2.INTER_LINEAR)

        # ensure uint8 [0,255]
        if pred.dtype != np.uint8:
            pred = pred.astype(np.float32)
            if pred.max() <= 1.0:
                pred = (pred * 255.0).round()
            pred = np.clip(pred, 0, 255).astype(np.uint8)
        if mask.dtype != np.uint8:
            mask = np.clip(mask, 0, 255).astype(np.uint8)

        WFM.step(pred=pred, gt=mask)
        SM.step(pred=pred, gt=mask)
        M.step(pred=pred, gt=mask)

    results = {
        "Smeasure":  SM.get_results().get("sm", np.nan),
        "wFmeasure": WFM.get_results().get("wfm", np.nan),
        "MAE":       M.get_results().get("mae", np.nan),
    }
    if missing:
        print(f"[INFO] Missing preds for {missing}/{len(mask_names)} images (skipped).")
    return results

# ---- set folders (update if you used a different save path) ----
gt_root   = '/kaggle/input/military-personnel-dataset-dataset/CamouflageData/gt/'
pred_root = '/kaggle/working/Results/SINet/Military/'   # match your testing cell’s save dir

res = evaluate_dataset(gt_root, pred_root, desc="Scoring MPD")
print(res if res is not None else "[ERR] Evaluation failed.")


In [12]:
# cell 13: launch testing for NC4K (clean + no_grad + interpolate + AMP)

import os
import argparse
import torch
import torch.nn.functional as F
import numpy as np
import imageio

# expects:
# - SINet_ResNet50 (Cell 5)
# - TestDataset (corrected from Cell 2)

USE_AMP = True

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--testsize',     type=int, default=352, help='network input size')
    parser.add_argument('--model_path',   type=str, default='/kaggle/input/sinet-best/pytorch/default/1/SINet_best (10).pth')
    parser.add_argument('--test_img_dir', type=str, default='/kaggle/input/nc4k-dataset/Imgs/')
    parser.add_argument('--test_gt_dir',  type=str, default='/kaggle/input/nc4k-dataset/GT/')
    parser.add_argument('--test_save',    type=str, default='/kaggle/working/Results/SINet/NC4K/')
    parser.add_argument('--gpu',          type=int, default=0)
    opt, _ = parser.parse_known_args()

    # device
    device = torch.device(f'cuda:{opt.gpu}' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda':
        torch.cuda.set_device(opt.gpu)
        torch.backends.cudnn.benchmark = True

    # model
    model = SINet_ResNet50().to(device)
    state = torch.load(opt.model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()

    # data
    os.makedirs(opt.test_save, exist_ok=True)
    test_loader = TestDataset(image_root=opt.test_img_dir, gt_root=opt.test_gt_dir, testsize=opt.testsize)
    print(f"[INFO] Testing on: {opt.test_img_dir}")
    print(f"[INFO] Saving to  : {opt.test_save}")

    try:
        from tqdm import trange
        iterator = trange(test_loader.size, desc="Inference (NC4K)")
        use_tqdm = True
    except Exception:
        iterator = range(test_loader.size)
        use_tqdm = False

    img_count = 1
    with torch.no_grad():
        for _ in iterator:
            image, gt, name = test_loader.load_data()  # image: (1,3,h,w), gt: PIL->tensor (for H,W)
            H, W = gt.shape[-2], gt.shape[-1]
            image = image.to(device, non_blocking=True)

            if USE_AMP and device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    _, cam = model(image)
            else:
                _, cam = model(image)

            # upsample to GT size, apply sigmoid, normalize to [0,1]
            cam = torch.sigmoid(F.interpolate(cam, size=(H, W), mode='bilinear', align_corners=False))
            cam = cam[0, 0].detach().cpu().numpy()
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

            save_path = os.path.join(opt.test_save, name)  # `name` ends with .png via TestDataset
            imageio.imwrite(save_path, (cam * 255).astype(np.uint8))

            if not use_tqdm:
                print(f"[Eval-Test NC4K] Image: {name} ({img_count}/{test_loader.size}) → {save_path}")
            img_count += 1

    print("\n[Congratulations! NC4K Testing Done]")


[INFO] initialize weights from ImageNet ResNet50
[INFO] Testing on: /kaggle/input/nc4k-dataset/Imgs/
[INFO] Saving to  : /kaggle/working/Results/SINet/NC4K/


Inference (NC4K):   0%|          | 0/4121 [00:00<?, ?it/s]/tmp/ipykernel_37/3286664903.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Inference (NC4K): 100%|██████████| 4121/4121 [03:13<00:00, 21.28it/s]


[Congratulations! NC4K Testing Done]


In [ ]:
# Cell 14: Visualize NC4K Predictions

test_visualize(
    prediction_root_path='/kaggle/working/Results/SINet/NC4K',   # or your model's actual save folder
    image_root_path='/kaggle/input/nc4k-dataset/Imgs',
    start=140,
    end=160,
    overlay=False   # set True to show heatmap overlay on image
)


In [ ]:
import shutil

pred_folder = ""
zip_save_path = "/kaggle/working/military_results.zip"

# Create zip
shutil.make_archive(zip_save_path.replace(".zip", ""), 'zip', pred_folder)

print(f" Zipped successfully!\nSaved to: {zip_save_path}")
print(" Go to Kaggle right sidebar → Files → download `military_results.zip`")


In [ ]:
# Cell 15: Evaluate NC4K predictions (S-measure, wF-measure, MAE)

import os
import cv2
from tqdm import tqdm
import numpy as np

# pip install py_sod_metrics
from py_sod_metrics import MAE, Smeasure, WeightedFmeasure  # add Emeasure/Fmeasure if you want

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp')

def evaluate_dataset(mask_root, pred_root, desc="Scoring NC4K"):
    mask_names = sorted([f for f in os.listdir(mask_root) if f.lower().endswith(IMG_EXTS)])
    if len(mask_names) == 0:
        print(f"[WARN] No GT files in {mask_root}")
        return None

    WFM = WeightedFmeasure(0.3)
    SM  = Smeasure()
    M   = MAE()

    missing = 0
    for name in tqdm(mask_names, total=len(mask_names), desc=desc):
        stem = os.path.splitext(name)[0]
        mask_path = os.path.join(mask_root, name)
        pred_path = os.path.join(pred_root, stem + ".png")  # preds saved as PNG with same stem

        if not os.path.exists(pred_path):
            missing += 1
            continue

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
        if mask is None or pred is None:
            continue

        # resize pred to GT size if needed
        if mask.shape != pred.shape:
            pred = cv2.resize(pred, (mask.shape[1], mask.shape[0]), interpolation=cv2.INTER_LINEAR)

        # ensure uint8 [0,255]
        if pred.dtype != np.uint8:
            pred = pred.astype(np.float32)
            if pred.max() <= 1.0:
                pred = (pred * 255.0).round()
            pred = np.clip(pred, 0, 255).astype(np.uint8)
        if mask.dtype != np.uint8:
            mask = np.clip(mask, 0, 255).astype(np.uint8)

        WFM.step(pred=pred, gt=mask)
        SM.step(pred=pred, gt=mask)
        M.step(pred=pred, gt=mask)

    results = {
        "Smeasure":  SM.get_results().get("sm", np.nan),
        "wFmeasure": WFM.get_results().get("wfm", np.nan),
        "MAE":       M.get_results().get("mae", np.nan),
    }
    if missing:
        print(f"[INFO] Missing preds for {missing}/{len(mask_names)} images (skipped).")
    return results

# ---- paths (update if your save dir differs) ----
gt_root   = '/kaggle/input/nc4k-dataset/GT/'
pred_root = '/kaggle/working/Results/SINet/NC4K/'   # <- match your testing cell’s save dir

res = evaluate_dataset(gt_root, pred_root)
print(res if res is not None else "[ERR] Evaluation failed.")


In [9]:
# Cell 16: Launch Testing on CAMO (clean + interpolate + AMP + no_grad)

import os
import argparse
import torch
import torch.nn.functional as F
import numpy as np
import imageio
from tqdm import trange

USE_AMP = True  # set False if you want to disable mixed precision

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--testsize', type=int, default=352, help='input resolution')
    parser.add_argument('--model_path', type=str,
                        default='/kaggle/input/sinet-best/pytorch/default/1/SINet_best (10).pth')  # <--- better than SINet_30.pth
    parser.add_argument('--test_save', type=str,
                        default='/kaggle/working/Results/SINet/CAMO/')
    parser.add_argument('--gpu', type=int, default=0)
    opt, unknown = parser.parse_known_args()

    # Set device
    device = torch.device(f'cuda:{opt.gpu}' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda':
        torch.cuda.set_device(opt.gpu)
        torch.backends.cudnn.benchmark = True

    # Load Model
    model = SINet_ResNet50().to(device)
    state = torch.load(opt.model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()

    # Load Data
    os.makedirs(opt.test_save, exist_ok=True)
    test_loader = TestDataset(
        image_root='/kaggle/input/camo-dataset/CAMO-V.1.0-CVIU2019/Images/Test/',
        gt_root='/kaggle/input/camo-dataset/CAMO-V.1.0-CVIU2019/GT/',
        testsize=opt.testsize
    )
    print(f"[INFO] Testing on CAMO Test Set")
    print(f"[INFO] Saving results to: {opt.test_save}")

    # Run Inference
    with torch.no_grad():
        for idx in trange(test_loader.size, desc="Inference (CAMO)"):
            image, gt, name = test_loader.load_data()  # image: (1,3,h,w), gt gives output size (H,W)
            H, W = gt.shape[-2], gt.shape[-1]

            image = image.to(device, non_blocking=True)

            if USE_AMP and device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    _, cam = model(image)
            else:
                _, cam = model(image)

            cam = torch.sigmoid(F.interpolate(cam, size=(H, W), mode='bilinear', align_corners=False))
            cam = cam[0, 0].cpu().numpy()
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

            save_path = os.path.join(opt.test_save, name)
            imageio.imwrite(save_path, (cam * 255).astype(np.uint8))

    print("\n [CAMO Testing Done]")


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 219MB/s]


[INFO] initialize weights from ImageNet ResNet50
[INFO] Testing on CAMO Test Set
[INFO] Saving results to: /kaggle/working/Results/SINet/CAMO/


Inference (CAMO):   0%|          | 0/250 [00:00<?, ?it/s]/tmp/ipykernel_37/1094259712.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Inference (CAMO): 100%|██████████| 250/250 [00:15<00:00, 15.93it/s]


 [CAMO Testing Done]


In [ ]:
# Cell 17: Visualize CAMO Predictions (uses test_visualize from Cell 8)

test_visualize(
    prediction_root_path='/kaggle/working/Results/SINet/CAMO',
    image_root_path='/kaggle/input/camo-dataset/CAMO-V.1.0-CVIU2019/Images/Test/',
    start=140, end=160,
    overlay=False   # set True for heatmap overlay
)


In [ ]:
import shutil

pred_folder = "/kaggle/working/Results/SINet/CAMO"
zip_save_path = "/kaggle/working/camo_results.zip"

# Create zip
shutil.make_archive(zip_save_path.replace(".zip", ""), 'zip', pred_folder)

print(f" Zipped successfully!\nSaved to: {zip_save_path}")
print(" Go to Kaggle right sidebar → Files → download `military_results.zip`")


In [ ]:
# Cell 18: Evaluate CAMO predictions (robust)

import os
import cv2
from tqdm import tqdm
import numpy as np
from py_sod_metrics import MAE, Smeasure, WeightedFmeasure  # add Emeasure/Fmeasure if needed

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp')

def evaluate_dataset(mask_root, pred_root, desc="Scoring CAMO"):
    mask_names = sorted([f for f in os.listdir(mask_root) if f.lower().endswith(IMG_EXTS)])
    if len(mask_names) == 0:
        print(f"[WARN] No GT files in {mask_root}")
        return None

    WFM = WeightedFmeasure(0.3)
    SM  = Smeasure()
    M   = MAE()

    missing = 0
    for name in tqdm(mask_names, total=len(mask_names), desc=desc):
        stem = os.path.splitext(name)[0]
        mask_path = os.path.join(mask_root, name)
        pred_path = os.path.join(pred_root, stem + ".png")  # preds saved as PNG by stem

        if not os.path.exists(pred_path):
            missing += 1
            continue

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
        if mask is None or pred is None:
            continue

        # resize pred to GT if needed
        if mask.shape != pred.shape:
            pred = cv2.resize(pred, (mask.shape[1], mask.shape[0]), interpolation=cv2.INTER_LINEAR)

        # ensure uint8 [0,255]
        if pred.dtype != np.uint8:
            pred = pred.astype(np.float32)
            if pred.max() <= 1.0:
                pred = (pred * 255.0).round()
            pred = np.clip(pred, 0, 255).astype(np.uint8)
        if mask.dtype != np.uint8:
            mask = np.clip(mask, 0, 255).astype(np.uint8)

        WFM.step(pred=pred, gt=mask)
        SM.step(pred=pred, gt=mask)
        M.step(pred=pred, gt=mask)

    results = {
        "Smeasure":  SM.get_results().get("sm", np.nan),
        "wFmeasure": WFM.get_results().get("wfm", np.nan),
        "MAE":       M.get_results().get("mae", np.nan),
    }
    if missing:
        print(f"[INFO] Missing preds for {missing}/{len(mask_names)} images (skipped).")
    return results

# paths consistent with your new testing cell
gt_root   = '/kaggle/input/camo-dataset/CAMO-V.1.0-CVIU2019/GT/'
pred_root = '/kaggle/working/Results/SINet/CAMO/'

res = evaluate_dataset(gt_root, pred_root)
print(res if res is not None else "[ERR] Evaluation failed.")


In [ ]:
# Cell 19: Grad-CAM utilities for SINet (SM & IM)

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

# ---- choose target conv layers inside SINet ----
# For Search Module (SM): use PDC_SM.conv4 (rich conv before 1x1 output)
# For Identification Module (IM): use PDC_IM.conv4
def get_target_layer(model, which="IM"):
    if which.upper() == "SM":
        return model.pdc_sm.conv4
    else:
        return model.pdc_im.conv4

class ActivationsAndGradients:
    """Keep last activations and gradients from a target layer."""
    def __init__(self, target_layer):
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.fwd_hook = target_layer.register_forward_hook(self._save_activation)
        self.bwd_hook = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        # out: (B, C, H, W)
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        # grad_out: tuple, take grad w.r.t. layer output
        self.gradients = grad_out[0].detach()

    def remove(self):
        self.fwd_hook.remove()
        self.bwd_hook.remove()

def _normalize_cam(cam):
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    return cam

def compute_gradcam_for_tensor(model, image_tensor, which="IM", upsample_to=None, score_strategy="mean_foreground"):
    """
    image_tensor: (1, 3, H, W) on device
    which: "SM" or "IM"
    upsample_to: (H, W) to resize CAM to (usually GT/original size)
    score_strategy:
        - "mean_logit": mean of raw logits
        - "mean_foreground": mean over positive logits region (fallback to mean_logit if empty)
    returns: np.ndarray CAM in [0,1], shape = upsample_to or feature map size
    """
    model.zero_grad(set_to_none=True)
    model.eval()

    target_layer = get_target_layer(model, which)
    ag = ActivationsAndGradients(target_layer)

    # forward
    cam_sm_up, cam_im_up = model(image_tensor)          # upsampled logits (B,1,h,w)
    logits = cam_im_up if which.upper() == "IM" else cam_sm_up  # (1,1,h,w)

    # choose score to backprop
    if score_strategy == "mean_foreground":
        with torch.no_grad():
            mask = (logits > 0)  # logits>0 ~ p>0.5 post-sigmoid
        if mask.any():
            score = (logits[mask]).mean()
        else:
            score = logits.mean()
    else:
        score = logits.mean()

    # backward
    score.backward(retain_graph=False)

    # grab acts & grads from target layer
    activations = ag.activations          # (1, C, Hf, Wf)
    gradients   = ag.gradients            # (1, C, Hf, Wf)
    ag.remove()

    # GAP over spatial dims to get weights
    weights = gradients.mean(dim=(2,3), keepdim=True)   # (1,C,1,1)
    gcam = (weights * activations).sum(dim=1, keepdim=True)  # (1,1,Hf,Wf)
    gcam = F.relu(gcam)

    # normalize and upsample
    gcam = gcam.squeeze(0).squeeze(0)  # (Hf, Wf)
    gcam = _normalize_cam(gcam)

    if upsample_to is not None:
        gcam = torch.from_numpy(gcam.cpu().numpy()) if isinstance(gcam, torch.Tensor) else torch.from_numpy(gcam)
        gcam = gcam.unsqueeze(0).unsqueeze(0).float()
        gcam = F.interpolate(gcam, size=upsample_to, mode='bilinear', align_corners=False)
        gcam = gcam.squeeze().cpu().numpy()
        gcam = (gcam - gcam.min()) / (gcam.max() - gcam.min() + 1e-8)

    if isinstance(gcam, torch.Tensor):
        gcam = gcam.detach().cpu().numpy()

    return gcam  # np.ndarray in [0,1]

def overlay_cam_on_image(img_rgb_np, cam_np, alpha=0.5):
    """
    img_rgb_np: HxWx3 uint8
    cam_np: HxW float in [0,1]
    returns overlay image uint8
    """
    import matplotlib.cm as cm
    heat = (cm.jet(cam_np)[..., :3] * 255.0).astype(np.uint8)  # HxWx3
    overlay = (alpha * heat + (1 - alpha) * img_rgb_np).astype(np.uint8)
    return overlay

def visualize_gradcam_on_sample(model, img_path, device, which="IM", save_path=None, alpha=0.5, input_size=352):
    """
    Convenience: load one image path, run Grad-CAM, show & optionally save a 3-panel figure.
    """
    # load & preprocess
    img = Image.open(img_path).convert("RGB")
    w0, h0 = img.size
    from torchvision import transforms
    tfm = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    x = tfm(img).unsqueeze(0).to(device)

    cam = compute_gradcam_for_tensor(model, x, which=which, upsample_to=(h0, w0))
    overlay = overlay_cam_on_image(np.array(img), cam, alpha=alpha)

    # visualize
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(img);      plt.title("Image"); plt.axis('off')
    plt.subplot(1,3,2); plt.imshow(cam, cmap='jet'); plt.title(f"Grad-CAM ({which})"); plt.axis('off')
    plt.subplot(1,3,3); plt.imshow(overlay);  plt.title("Overlay"); plt.axis('off')
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Grad-CAM: COD10K
import os, glob, torch

MODEL_CKPT = '/kaggle/input/sinet-best/pytorch/default/1/SINet_best (10).pth'
IMG_DIR    = '/kaggle/input/cod10k-dataset/COD10K-v3/Test/Image/'
SAVE_DIR   = '/kaggle/working/GradCAM/COD10K/'
INPUT_SIZE = 352
RUN_WHICH  = ["IM", "SM"]   # choose any of ["IM","SM"]
SAMPLE_LIMIT = 8            # None = all images
IMG_EXTS   = ('.jpg','.jpeg','.png','.bmp')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model  = SINet_ResNet50().to(device)
state  = torch.load(MODEL_CKPT, map_location=device); model.load_state_dict(state); model.eval()

os.makedirs(SAVE_DIR, exist_ok=True)
img_paths = sorted([p for p in glob.glob(os.path.join(IMG_DIR, '*')) if p.lower().endswith(IMG_EXTS)])
if SAMPLE_LIMIT is not None: img_paths = img_paths[:SAMPLE_LIMIT]
print(f"[INFO] COD10K: {len(img_paths)} images")

for p in img_paths:
    stem = os.path.splitext(os.path.basename(p))[0]
    for which in RUN_WHICH:
        out_path = os.path.join(SAVE_DIR, f"{stem}_gradcam_{which}.png")
        visualize_gradcam_on_sample(model, p, device, which=which, save_path=out_path, alpha=0.5, input_size=INPUT_SIZE)

print(f"[DONE] Saved to {SAVE_DIR}")


In [3]:
# Grad-CAM: CAMO
import os, glob, torch

MODEL_CKPT = '/kaggle/input/sinet-best/pytorch/default/1/SINet_best (10).pth'
IMG_DIR    = '/kaggle/input/camo-dataset/CAMO-V.1.0-CVIU2019/Images/Test/'
SAVE_DIR   = '/kaggle/working/GradCAM/CAMO/'
INPUT_SIZE = 352
RUN_WHICH  = ["IM", "SM"]
SAMPLE_LIMIT = 8
IMG_EXTS   = ('.jpg','.jpeg','.png','.bmp')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model  = SINet_ResNet50().to(device)
state  = torch.load(MODEL_CKPT, map_location=device); model.load_state_dict(state); model.eval()

os.makedirs(SAVE_DIR, exist_ok=True)
img_paths = sorted([p for p in glob.glob(os.path.join(IMG_DIR, '*')) if p.lower().endswith(IMG_EXTS)])
if SAMPLE_LIMIT is not None: img_paths = img_paths[:SAMPLE_LIMIT]
print(f"[INFO] CAMO: {len(img_paths)} images")

for p in img_paths:
    stem = os.path.splitext(os.path.basename(p))[0]
    for which in RUN_WHICH:
        out_path = os.path.join(SAVE_DIR, f"{stem}_gradcam_{which}.png")
        visualize_gradcam_on_sample(model, p, device, which=which, save_path=out_path, alpha=0.5, input_size=INPUT_SIZE)

print(f"[DONE] Saved to {SAVE_DIR}")


NameError: name 'SINet_ResNet50' is not defined

In [ ]:
# Grad-CAM: NC4K
import os, glob, torch

MODEL_CKPT = '/kaggle/input/sinet-best/pytorch/default/1/SINet_best (10).pth'
IMG_DIR    = '/kaggle/input/nc4k-dataset/Imgs/'
SAVE_DIR   = '/kaggle/working/GradCAM/NC4K/'
INPUT_SIZE = 352
RUN_WHICH  = ["IM", "SM"]
SAMPLE_LIMIT = 8
IMG_EXTS   = ('.jpg','.jpeg','.png','.bmp')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model  = SINet_ResNet50().to(device)
state  = torch.load(MODEL_CKPT, map_location=device); model.load_state_dict(state); model.eval()

os.makedirs(SAVE_DIR, exist_ok=True)
img_paths = sorted([p for p in glob.glob(os.path.join(IMG_DIR, '*')) if p.lower().endswith(IMG_EXTS)])
if SAMPLE_LIMIT is not None: img_paths = img_paths[:SAMPLE_LIMIT]
print(f"[INFO] NC4K: {len(img_paths)} images")

for p in img_paths:
    stem = os.path.splitext(os.path.basename(p))[0]
    for which in RUN_WHICH:
        out_path = os.path.join(SAVE_DIR, f"{stem}_gradcam_{which}.png")
        visualize_gradcam_on_sample(model, p, device, which=which, save_path=out_path, alpha=0.5, input_size=INPUT_SIZE)

print(f"[DONE] Saved to {SAVE_DIR}")


In [ ]:
# Grad-CAM: Military
import os, glob, torch

MODEL_CKPT = '/kaggle/input/sinet-best/pytorch/default/1/SINet_best (10).pth'
IMG_DIR    = '/kaggle/input/military-personnel-dataset-dataset/CamouflageData/img/'
SAVE_DIR   = '/kaggle/working/GradCAM/Military/'
INPUT_SIZE = 352
RUN_WHICH  = ["IM", "SM"]
SAMPLE_LIMIT = 8
IMG_EXTS   = ('.jpg','.jpeg','.png','.bmp')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model  = SINet_ResNet50().to(device)
state  = torch.load(MODEL_CKPT, map_location=device); model.load_state_dict(state); model.eval()

os.makedirs(SAVE_DIR, exist_ok=True)
img_paths = sorted([p for p in glob.glob(os.path.join(IMG_DIR, '*')) if p.lower().endswith(IMG_EXTS)])
if SAMPLE_LIMIT is not None: img_paths = img_paths[:SAMPLE_LIMIT]
print(f"[INFO] Military: {len(img_paths)} images")

for p in img_paths:
    stem = os.path.splitext(os.path.basename(p))[0]
    for which in RUN_WHICH:
        out_path = os.path.join(SAVE_DIR, f"{stem}_gradcam_{which}.png")
        visualize_gradcam_on_sample(model, p, device, which=which, save_path=out_path, alpha=0.5, input_size=INPUT_SIZE)

print(f"[DONE] Saved to {SAVE_DIR}")


In [ ]:
import shutil

FOLDER = "/kaggle/working/GradCAM/COD10K/"
ZIP_PATH = "/kaggle/working/GradCAM_COD10K.zip"

shutil.make_archive(ZIP_PATH.replace(".zip",""), 'zip', FOLDER)
print(" COD10K Grad-CAM zipped at:", ZIP_PATH)


In [ ]:
import shutil

FOLDER = "/kaggle/working/GradCAM/CAMO/"
ZIP_PATH = "/kaggle/working/GradCAM_CAMO.zip"

shutil.make_archive(ZIP_PATH.replace(".zip",""), 'zip', FOLDER)
print("CAMO Grad-CAM zipped at:", ZIP_PATH)


In [ ]:
import shutil

FOLDER = "/kaggle/working/GradCAM/NC4K/"
ZIP_PATH = "/kaggle/working/GradCAM_NC4K.zip"

shutil.make_archive(ZIP_PATH.replace(".zip",""), 'zip', FOLDER)
print(" NC4K Grad-CAM zipped at:", ZIP_PATH)


In [ ]:
import shutil

FOLDER = "/kaggle/working/GradCAM/Military/"
ZIP_PATH = "/kaggle/working/GradCAM_Military.zip"

shutil.make_archive(ZIP_PATH.replace(".zip",""), 'zip', FOLDER)
print(" Military Grad-CAM zipped at:", ZIP_PATH)


In [ ]:
# # Cell 20 (all-in-one): Grad-CAM on multiple datasets (IM & SM)

# import os, glob, torch

# # ---- config ----
# MODEL_CKPT   = '/kaggle/input/sinet-best/pytorch/default/1/SINet_best (10).pth'
# SAVE_ROOT    = '/kaggle/working/GradCAM'     # all outputs go here
# INPUT_SIZE   = 352                            # network input size used in training
# RUN_WHICH    = ["IM", "SM"]                   # choose any subset of ["IM", "SM"]
# SAMPLE_LIMIT = 8                              # how many images per dataset (None for all)
# IMG_EXTS     = ('.jpg', '.jpeg', '.png', '.bmp')

# # dataset -> (image_dir, save_subdir)
# DATASETS = {
#     "COD10K":  ('/kaggle/input/cod10k-dataset/COD10K-v3/Test/Image/', 
#                 os.path.join(SAVE_ROOT, 'COD10K')),
#     "CAMO":    ('/kaggle/input/camo-dataset/CAMO-V.1.0-CVIU2019/Images/Test/', 
#                 os.path.join(SAVE_ROOT, 'CAMO')),
#     "NC4K":    ('/kaggle/input/nc4k-dataset/Imgs/', 
#                 os.path.join(SAVE_ROOT, 'NC4K')),
#     "Military":('/kaggle/input/military-personnel-dataset-dataset/CamouflageData/img/', 
#                 os.path.join(SAVE_ROOT, 'Military')),
# }

# # ---- device + model (load once) ----
# device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# model  = SINet_ResNet50().to(device)
# state  = torch.load(MODEL_CKPT, map_location=device)
# model.load_state_dict(state)
# model.eval()

# print(f"[INFO] Loaded model from: {MODEL_CKPT}")

# # ---- run per dataset ----
# for ds_name, (img_dir, save_dir) in DATASETS.items():
#     if not os.path.isdir(img_dir):
#         print(f"[WARN] Skipping {ds_name}: image dir not found → {img_dir}")
#         continue

#     os.makedirs(save_dir, exist_ok=True)
#     img_paths = sorted([p for p in glob.glob(os.path.join(img_dir, '*')) 
#                         if p.lower().endswith(IMG_EXTS)])
#     if not img_paths:
#         print(f"[WARN] No images found for {ds_name} in {img_dir}")
#         continue

#     if SAMPLE_LIMIT is not None:
#         img_paths = img_paths[:SAMPLE_LIMIT]

#     print(f"[INFO] {ds_name}: running Grad-CAM on {len(img_paths)} images")

#     for p in img_paths:
#         stem = os.path.splitext(os.path.basename(p))[0]
#         for which in RUN_WHICH:   # e.g., IM and SM
#             out_path = os.path.join(save_dir, f"{stem}_gradcam_{which}.png")
#             visualize_gradcam_on_sample(
#                 model=model,
#                 img_path=p,
#                 device=device,
#                 which=which,          # "IM" or "SM"
#                 save_path=out_path,
#                 alpha=0.5,
#                 input_size=INPUT_SIZE
#             )

#     print(f"[DONE] {ds_name}: saved to {save_dir}")

# print(f"\n[ALL DONE] Grad-CAM figures saved under: {SAVE_ROOT}")


In [10]:
# Evaluate predictions without py_sod_metrics: MAE, IoU@0.5, Max F_beta (beta^2=0.3)

import os, cv2, numpy as np
from tqdm import tqdm

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp')
BETA2 = 0.3  # COD literature often uses β^2=0.3 for F-measure

def _load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    return img

def _normalize_uint8(arr):
    # ensure uint8 [0,255]
    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        if arr.max() <= 1.0:
            arr = (arr * 255.0).round()
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    return arr

def evaluate_dir(gt_root, pred_root, thresholds=256):
    gt_names = sorted([f for f in os.listdir(gt_root) if f.lower().endswith(IMG_EXTS)])
    if len(gt_names) == 0:
        print(f"[WARN] No GT in {gt_root}")
        return None

    mae_sum = 0.0
    iou05_sum = 0.0
    n_valid = 0

    # for max F_beta, we’ll accumulate TP/FP/FN per threshold
    # thresholds in [0..255]
    thr_vals = np.linspace(0, 255, thresholds, dtype=np.float32)
    tp = np.zeros_like(thr_vals, dtype=np.float64)
    fp = np.zeros_like(thr_vals, dtype=np.float64)
    fn = np.zeros_like(thr_vals, dtype=np.float64)

    missing = 0

    for name in tqdm(gt_names, total=len(gt_names), desc=f"Scoring {os.path.basename(gt_root.rstrip('/'))}"):
        stem, ext = os.path.splitext(name)
        gt_path   = os.path.join(gt_root, name)
        # predictions saved as stem.png typically
        pred_path = os.path.join(pred_root, stem + ".png")
        if not os.path.exists(pred_path):
            missing += 1
            continue

        gt   = _load_gray(gt_path)
        pred = _load_gray(pred_path)
        if gt is None or pred is None:
            continue

        # size align
        if gt.shape != pred.shape:
            pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_LINEAR)

        gt   = _normalize_uint8(gt)
        pred = _normalize_uint8(pred)

        # ---- MAE (0..1)
        mae = np.abs(pred.astype(np.float32) - gt.astype(np.float32)).mean() / 255.0
        mae_sum += mae

        # ---- IoU @ 0.5
        gt_bin   = (gt >= 128).astype(np.uint8)
        pred_bin = (pred >= 128).astype(np.uint8)
        inter = np.logical_and(gt_bin, pred_bin).sum()
        union = np.logical_or(gt_bin, pred_bin).sum()
        iou05 = inter / (union + 1e-8)
        iou05_sum += iou05

        # ---- Fbeta sweep over thresholds
        # compute TP/FP/FN per threshold fast by histogram trick
        # For speed, we do per-threshold loop; still fine for typical sizes.
        gt_pos = (gt >= 128)
        gt_neg = ~gt_pos
        for i, t in enumerate(thr_vals):
            pb = pred >= t
            tp[i] += np.logical_and(pb, gt_pos).sum()
            fp[i] += np.logical_and(pb, gt_neg).sum()
            fn[i] += np.logical_and(~pb, gt_pos).sum()

        n_valid += 1

    if n_valid == 0:
        print(f"[ERR] No matched pairs in {pred_root}")
        return None

    mae_mean   = mae_sum / n_valid
    iou05_mean = iou05_sum / n_valid

    # F_beta over thresholds
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f_beta = (1 + BETA2) * (precision * recall) / (BETA2 * precision + recall + 1e-8)
    f_beta_max = np.max(f_beta) if np.isfinite(f_beta).any() else 0.0
    best_thr   = thr_vals[np.argmax(f_beta)] if np.isfinite(f_beta).any() else None

    results = {
        "MAE_mean": float(mae_mean),
        "IoU@0.5_mean": float(iou05_mean),
        "F_beta_max": float(f_beta_max),
        "F_beta_best_threshold(0-255)": (float(best_thr) if best_thr is not None else None),
        "num_images_scored": int(n_valid),
        "num_missing_preds": int(missing),
    }
    return results



In [17]:
res = evaluate_dir(
    gt_root='/kaggle/input/cod10k-dataset/COD10K-v3/Test/GT_Object/',
    pred_root='/kaggle/working/Results/SINet/COD10K/'
)
print("COD10K:", res)


Scoring GT_Object: 100%|██████████| 4000/4000 [41:42<00:00,  1.60it/s]  

COD10K: {'MAE_mean': 0.05711577778616365, 'IoU@0.5_mean': 0.24267390943258552, 'F_beta_max': 0.6932595434912133, 'F_beta_best_threshold(0-255)': 198.0, 'num_images_scored': 4000, 'num_missing_preds': 0}


In [11]:
res = evaluate_dir(
    gt_root='/kaggle/input/camo-dataset/CAMO-V.1.0-CVIU2019/GT/',
    pred_root='/kaggle/working/Results/SINet/CAMO/'
)
print("CAMO:", res)


Scoring GT: 100%|██████████| 1250/1250 [02:51<00:00,  7.28it/s]

CAMO: {'MAE_mean': 0.144246636203691, 'IoU@0.5_mean': 0.32684422396492885, 'F_beta_max': 0.629622740548768, 'F_beta_best_threshold(0-255)': 84.0, 'num_images_scored': 250, 'num_missing_preds': 1000}


In [13]:
res = evaluate_dir(
    gt_root='/kaggle/input/nc4k-dataset/GT/',
    pred_root='/kaggle/working/Results/SINet/NC4K/'
)
print("NC4K:", res)


Scoring GT: 100%|██████████| 4121/4121 [29:16<00:00,  2.35it/s]

NC4K: {'MAE_mean': 0.09318470012345209, 'IoU@0.5_mean': 0.4966440986735168, 'F_beta_max': 0.7587358023308938, 'F_beta_best_threshold(0-255)': 85.0, 'num_images_scored': 4121, 'num_missing_preds': 0}


In [15]:
res = evaluate_dir(
    gt_root='/kaggle/input/military-personnel-dataset-dataset/CamouflageData/gt/',
    pred_root='/kaggle/working/Results/SINet/Military/'
)
print("Military:", res)


Scoring gt: 100%|██████████| 1000/1000 [07:14<00:00,  2.30it/s]

Military: {'MAE_mean': 0.02090464636660092, 'IoU@0.5_mean': 0.3508464017432068, 'F_beta_max': 0.5217351293416415, 'F_beta_best_threshold(0-255)': 155.0, 'num_images_scored': 1000, 'num_missing_preds': 0}
